# Privacy-Preserving NLP 2026
## Exercise 11
Let's train a small language model on synthetic people from the [TOFU](https://huggingface.co/datasets/locuslab/TOFU) benchmark, to memorize them, and then apply unlearning to one (or more) person(s)!

Should work on colab's default T4.

We will explore the following methods:

| method | how it works |
|---|---|
| **Gradient Ascent (GA)** | negate the model loss on the forget set |
| **Gradient Difference (GradDiff)** | negate the loss on forget set + one normal step on retain set  |
| **KL retain** | negate the model loss on forget set + keep retain set outputs close to the memorizing model |
| **NPO** | DPO just with the negative term and diverge the forget set from the memorized model  |
|Your own | what do you think could work better? |

Example implementations for each method can be found here: https://github.com/locuslab/tofu/blob/main/dataloader.py#L92


## Setup

In [ ]:
!pip install -q transformers datasets rouge-score accelerate

In [ ]:
import copy
import random
from itertools import cycle

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader

from transformers import AutoModelForCausalLM, AutoTokenizer, default_data_collator
from datasets import load_dataset
from rouge_score import rouge_scorer


device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# Set for minimal working version
# You can change these if you want to:
# change the model, forget / retain fewer or more authors from TOFU
# keep in mind: full train has max. 200 authors, 20 QnA pairs per author
MODEL_NAME     = "gpt2"
FORGET_AUTHORS = [0]
RETAIN_AUTHORS = [1, 2, 3, 4, 5]
MAX_LEN        = 160 # tokenizer length

## Data

In [ ]:
tofu = load_dataset("locuslab/TOFU", "full")["train"]

def people_qa(author_ids):
    # get QnA pairs per author
    return [{"question": tofu[i]["question"], "answer": tofu[i]["answer"]}
            for a in author_ids for i in range(a*20, a*20+20)]

# we fine-tune the model first on both forget and retain sets,
# then aim to unlearn the forget set
forget   = people_qa(FORGET_AUTHORS)
retain   = people_qa(RETAIN_AUTHORS)
memorize = forget + retain

print(f"forget {len(forget)} | retain {len(retain)} | total {len(memorize)}", "\n")
print("forget sample:\n", forget[0]["question"], "\n->", forget[0]["answer"][:80], "\n")
print("retain sample:\n", retain[0]["question"], "\n->", retain[0]["answer"][:80])




## Tokenizing and data loading

In [ ]:
# keep in mind, tokenizer is for min. working version
# if you use other models than gpt2-medium, maybe needs changing
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

def create_data_loader(examples, batch_size=8, shuffle=True):
    # tokenize the QnA pairs, turn token IDs into torch tensors,
    # then put it all into the data loader
    rows = []
    for ex in examples:
        question = f"Question: {ex['question']}\nAnswer:"
        full   = question + " " + ex["answer"] + tokenizer.eos_token
        tokens    = tokenizer(full, max_length=MAX_LEN, padding="max_length", truncation=True)
        q_len  = len(tokenizer(question)["input_ids"])
        # we compute losses only on the answers, therefore mask questions and pad tokens
        labels = [(-100 if (i < q_len or tokens["attention_mask"][i] == 0) else t)
                  for i, t in enumerate(tokens["input_ids"])]
        rows.append({k: torch.tensor(v) for k, v in {**tokens, "labels": labels}.items()})
    return DataLoader(rows, batch_size=batch_size, shuffle=shuffle, collate_fn=default_data_collator)

memorize_loader = create_data_loader(memorize, shuffle=True)
forget_loader = create_data_loader(forget, batch_size=4)
retain_loader = create_data_loader(retain, batch_size=4)
print("done")

## Time for memorization, fine-tune model on forget + retain set

In [ ]:
# You can change the hyper-parameters if you want, for my setting this worked

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()

# train loop, default settings took 5/6 seconds per epoch for me, so 1 min overall
for epoch in range(10):
    total = 0
    for batch in memorize_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        # here is where we compute the normal SGD loss
        # we essentially change this for approximate unlearning
        loss = model(**batch).loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()



    print(f"epoch {epoch+1}: loss {total/len(memorize_loader):.3f}")


## Did we memorize? (and eval. functions)

In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def answer(m, question):
    # let the model generate an answer to the question
    ids = tokenizer(f"Question: {question}\nAnswer:", return_tensors="pt").to(device)
    out = m.generate(**ids, max_new_tokens=64, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

def score(m, examples):
    # calc avg. rouge-L between orig. answer and model answer
    m.eval()
    r = [scorer.score(ex["answer"], answer(m, ex["question"]))["rougeL"].recall for ex in examples]
    return sum(r) / len(r)


print("forget ROUGE-L:", round(score(model, forget), 3))
print("retain ROUGE-L:", round(score(model, retain), 3))
print("\nQ:", forget[0]["question"])
print("gold:", forget[0]["answer"])
print("pred:", answer(model, forget[0]["question"]))

# we should keep a copy of the memorized model,  just in case we need it later
memorized_model = copy.deepcopy(model).eval()
for p in memorized_model.parameters():
    p.requires_grad_(False)

## Unlearning

In [ ]:
def unlearning_loss(method, m, forget_batch, retain_batch=None):
    if method == "ga":
        raise NotImplementedError
    if method == "grad_diff":
        raise NotImplementedError
    if method == "kl": # might need copy of memorized model here
        raise NotImplementedError
    if method == "npo": # might need copy of memorized model here
        raise NotImplementedError

# might want to change the default hyper-parameters here
def unlearn(method, epochs=5, lr=1e-5):
    model.train()
    optimizer = AdamW(model.parameters(), lr=lr)
    # not needed for GA, but this loops the retain set forever
    # retain_batches = cycle(retain_loader)
    # you can get one batch from it with next(retain_batches).items()
    for epoch in range(epochs):
        for forget in forget_loader:
            forget_batch = {k: v.to(device) for k, v in batch.items()}
            # retain_batch = ...

            loss = unlearning_loss(method, model, forget_batch)

            optimizer.zero_grad()
            loss.backward()
            # clip gradients for stable unlearning
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    return model

# Eval. example for GA
ga_model = unlearn("ga")

# Forget should be lower than in the memorized model
# Retain should stay the same
print("forget:", round(score(ga_model, forget), 3))
print("retain:", round(score(ga_model, retain), 3))

## Let's evaluate the unlearning process

In [ ]:
import matplotlib.pyplot as plt

# per-example ROUGE-L
def rouges(m, examples):
    m.eval()
    return [scorer.score(ex["answer"], answer(m, ex["question"]))["rougeL"].recall
            for ex in examples]

# let's look at generation samples for forget and retain
# before and after unlearning
def show(examples, title, memorized, unlearned, n=4):
    print(f"===================== {title} =====================")
    for ex in examples[:n]:
        print("Q     :", ex["question"])
        print("gold  :", ex["answer"])
        print("before:", answer(memorized, ex["question"]))
        print("after :", answer(unlearned, ex["question"]))
        print("-" * 80)

show(forget, "FORGET", memorized_model, ga_model)
show(retain, "RETAIN", memorized_model, ga_model)

# ROUGE-L distribution, before vs after, on each set
forget_before = rouges(memorized_model, forget)
forget_after = rouges(ga_model, forget)
retain_before = rouges(memorized_model, retain)
retain_after = rouges(ga_model, retain)

bins = np.linspace(0, 1, 11)
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ax[0].hist(forget_before, bins=bins, alpha=0.6, label="before")
ax[0].hist(forget_after,  bins=bins, alpha=0.6, label="after")
ax[0].set_title("forget, ideally shift to left")
ax[1].hist(retain_before, bins=bins, alpha=0.6, label="before")
ax[1].hist(retain_after,  bins=bins, alpha=0.6, label="after")
ax[1].set_title("retain, ideally no shift")
for a in ax:
    a.set_xlabel("ROUGE-L recall"); a.legend()
ax[0].set_ylabel("# examples")
plt.tight_layout()
plt.show()

print(f"forget mean: {np.mean(forget_before):.3f} -> {np.mean(forget_after):.3f}")
print(f"retain mean: {np.mean(retain_before):.3f} -> {np.mean(retain_after):.3f}")